In [1]:
from os import write
import random
import csv
from sympy import symbols, Eq, expand, simplify

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import pandas as pd
from matplotlib import pyplot as plt
import numpy as np

!pip install torchinfo
from torchinfo import summary

In [2]:
csv_path = "algebra_dataset (1).csv"

df = pd.read_csv(csv_path)
print(df.head())
print(df.shape)

     type       input_text target_text
0  linear  7x - 8 = -29 ->    7x = -21
1  linear      7x = -21 ->       x= -3
2  linear  9x - 2 = -65 ->    9x = -63
3  linear      9x = -63 ->       x= -7
4  linear    6x - 7 = 5 ->     6x = 12
(37541, 3)


In [3]:
def algebra_splitter(text):
    text = str(text).strip()
    tokens = []
    i = 0

    while i < len(text):
        ch = text[i]

        # handle arrow "->"
        if ch == '-' and i + 1 < len(text) and text[i + 1] == '>':
            tokens.append("->")
            i += 2
            continue

        # skip spaces
        if ch.isspace():
            i += 1
            continue

        # numbers (multi-digit)
        if ch.isdigit():
            num = ch
            i += 1
            while i < len(text) and text[i].isdigit():
                num += text[i]
                i += 1
            tokens.append(num)
            continue

        # variables (like x)
        if ch.isalpha():
            tokens.append(ch)
            i += 1
            continue

        # operators / parentheses
        if ch in "+-*/^=()":
            tokens.append(ch)
            i += 1
            continue

        # fallback (rare cases)
        tokens.append(ch)
        i += 1

    # add in explicit multiplication symbols
    new_tokens = []
    for j in range(len(tokens)):
        new_tokens.append(tokens[j])

        if j < len(tokens) - 1:
            a = tokens[j]
            b = tokens[j + 1]

            # cases where we want mult
            # x(, 2(, x x, 2 x, )x, )2
            if (
                (a.isdigit() and b.isalpha()) or
                (a.isalpha() and b.isalpha()) or
                (a in [")"] and (b.isalpha() or b.isdigit())) or
                ((a.isalpha() or a.isdigit()) and b in ["("])
            ):
                new_tokens.append("*")

    return new_tokens

In [4]:
def build_vocab(token_lists):
    # extra characters
    vocab = ["<PAD>", "<SOS>", "<EOS>", "<SEP>", "<UNK>"]

    for tokens in token_lists:
        for token in tokens:
            if token not in vocab:
                vocab.append(token)

    token_to_id = {token: i for i, token in enumerate(vocab)}
    id_to_token = {i: token for token, i in token_to_id.items()}

    return vocab, token_to_id, id_to_token

In [5]:
def encode(tokens, token_to_id):
    encoded = []

    for token in tokens:
        if token in token_to_id:
            encoded.append(token_to_id[token])
        else:
            encoded.append(token_to_id["<UNK>"])

    return encoded

In [6]:
def decode(ids, id_to_token):
    tokens = []

    for idx in ids:
        token = id_to_token[idx]

        if token == "<EOS>":
            break

        if token in ["<PAD>", "<SOS>"]:
            continue

        tokens.append(token)

    return tokens

In [7]:
example = "x(x - 9) = 0 ->"

split = algebra_splitter(example)
vocab, token_to_id, id_to_token = build_vocab([split])
tokens = encode(split, token_to_id)
decoded = decode(tokens, id_to_token)

token_df = pd.DataFrame({
    "Text": split,
    "Token": tokens
})

token_df.head(10)

,Text,Token
0,x,5
1,*,6
2,(,7
3,x,5
4,-,8
5,9,9
6,),10
7,=,11
8,0,12
9,->,13


In [135]:
class TargetOnlyNextTokenDataset(Dataset):
    def __init__(self, full_token_lists, token_to_id, context_length):
        self.data = []
        self.token_to_id = token_to_id
        self.context_length = context_length
        self.pad_id = token_to_id["<PAD>"]

        for tokens in full_token_lists:
            encoded = encode(tokens, token_to_id)

            sep_index = tokens.index("<SEP>")

            for i in range(sep_index + 1, len(encoded)):
                # train only on target-side tokens
                start = max(0, i - context_length)

                context = encoded[start:i]

                # left pad to fixed context length
                if len(context) < context_length:
                    context = [self.pad_id] * (context_length - len(context)) + context

                target = encoded[i]

                self.data.append((context, target))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        context, target = self.data[idx]
        return torch.tensor(context, dtype=torch.long), torch.tensor(target, dtype=torch.long)

In [132]:
csv_path = "algebra_dataset (2).csv"
df = pd.read_csv(csv_path)
full_token_lists = []

for inp, tgt in zip(df["input_text"], df["target_text"]):
    input_tokens = algebra_splitter(inp)
    target_tokens = algebra_splitter(tgt)

    full_tokens = ["<SOS>"] + input_tokens + ["<SEP>"] + target_tokens + ["<EOS>"]
    full_token_lists.append(full_tokens)

# build vocab
vocab, token_to_id, id_to_token = build_vocab(full_token_lists)

print("Number of equations:", len(full_token_lists))
print("Vocabulary size:", len(vocab))
print("First split example:", full_token_lists[0])


Number of equations: 60000
Vocabulary size: 117
First split example: ['<SOS>', '4', '*', 'x', '+', '1', '=', '33', '->', '<SEP>', '4', '*', 'x', '=', '32', '<EOS>']


In [128]:
class SeuussModel(nn.Module):
    def __init__(self, vocab_size, context_length, embedding_dim, hidden_dim):
        super().__init__()
        self.context_length = context_length
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.pipeline = nn.Sequential(
            nn.Linear(context_length * embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

    def forward(self, x):
        embedded = self.embedding(x)
        flattened = embedded.view(embedded.shape[0], -1)
        return self.pipeline(flattened)

In [133]:
# similar to previous model but with attention heads
class MathModel(nn.Module):
    def __init__(self, vocab_size, context_length, embedding_dim, hidden_dim):
        super().__init__()

        self.context_length = context_length
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=4,
            batch_first=True # I could not figure out the error not having this setting was causing and used chatGPT for assistance.
        )

        self.norm1 = nn.LayerNorm(embedding_dim)

        self.net = nn.Sequential(
            nn.Linear(context_length * embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

    def forward(self, x):
        embedded = self.embedding(x)

        attn_out, _ = self.attention(
            embedded,
            embedded,
            embedded
        )

        out = self.norm1(embedded + attn_out)

        flattened = out.reshape(out.shape[0], -1)

        return self.net(flattened)

In [234]:
# full transformer block model
class MathModel2(nn.Module):
  #https://www.geeksforgeeks.org/deep-learning/transformer-using-pytorch/
  #https://docs.pytorch.org/docs/2.11/generated/torch.nn.TransformerEncoderLayer.html?utm_source=chatgpt.com

    def __init__(
        self,
        vocab_size,
        context_length,
        embedding_dim=64,
        hidden_dim=256,
        num_heads=4,
        num_layers=2,
        pad_id=0,
        dropout=0.1
    ):
        super().__init__()

        self.context_length = context_length
        self.pad_id = pad_id

        # token embeddings
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)

        # positional embeddings
        self.position_embedding = nn.Embedding(context_length, embedding_dim)

        # transformer block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            batch_first=True,
            activation="relu"
        )

        # stack multiple transformer blocks
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(embedding_dim)

        # predict next token from final context position
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):

        batch_size, seq_len = x.shape

        positions = torch.arange(seq_len, device=x.device)
        positions = positions.unsqueeze(0).expand(batch_size, seq_len)

        token_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(positions)

        out = token_emb + pos_emb

        # mask padding tokens so attention does not focus on <PAD>
        padding_mask = x == self.pad_id

        out = self.transformer(
            out,
            src_key_padding_mask=padding_mask
        )

        out = self.norm(out)

        # use the last token in the context to predict the next token
        last_token = out[:, -1, :]

        logits = self.output(last_token)

        return logits

In [262]:
context_length = 8

vocab, token_to_id, id_to_token = build_vocab(full_token_lists)

dataset = TargetOnlyNextTokenDataset(
    full_token_lists,
    token_to_id,
    context_length=context_length
)

loader = DataLoader(dataset, batch_size=64, shuffle=True)

print("Training examples:", len(dataset))
print("Vocab size:", len(vocab))

Training examples: 560597
Vocab size: 117


In [265]:
model = MathModel2(
    vocab_size=len(vocab),
    context_length=context_length,
    embedding_dim=64,
    hidden_dim=128,
    num_heads=4,
    num_layers=2,
    pad_id=token_to_id["<PAD>"],
    dropout=0.1
).to(device)
summary(model)

Layer (type:depth-idx)                                            Param #
MathModel2                                                        --
├─Embedding: 1-1                                                  7,488
├─Embedding: 1-2                                                  2,048
├─TransformerEncoder: 1-3                                         --
│    └─ModuleList: 2-1                                            --
│    │    └─TransformerEncoderLayer: 3-1                          49,984
│    │    └─TransformerEncoderLayer: 3-2                          49,984
├─LayerNorm: 1-4                                                  128
├─Linear: 1-5                                                     7,605
Total params: 117,237
Trainable params: 117,237
Non-trainable params: 0

In [266]:
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

loss_history = []

for epoch in range(200):
    model.train()
    total_loss = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        opt.zero_grad()
        preds = model(X_batch)
        loss = loss_fn(preds, y_batch)
        loss.backward()
        opt.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1}: loss = {avg_loss:.4f}")

Epoch 1: loss = 0.4957
Epoch 2: loss = 0.4369
Epoch 3: loss = 0.4309
Epoch 4: loss = 0.4279
Epoch 5: loss = 0.4251
Epoch 6: loss = 0.4232
Epoch 7: loss = 0.4215
Epoch 8: loss = 0.4206
Epoch 9: loss = 0.4198
Epoch 10: loss = 0.4192


KeyboardInterrupt: 

In [267]:
def allowed_next_tokens(prev_token):
    numbers = {"0", "1", "2", "3", "4", "5", "6", "7", "8", "9"}

    # after an operator, equals sign, or separator,
    # the next token should usually start an expression.
    if prev_token in {"+", "-", "*", "/", "=", "<SEP>"}:
        return {"x", "("} | numbers

    # after a power symbol, usually expect a number.
    elif prev_token == "^":
        return numbers

    # after an open parenthesis, usually expect x or a number.
    elif prev_token == "(":
        return {"x"} | numbers

    else:
        return None


In [268]:
def filter_logits_by_allowed_tokens(logits, prev_token, token_to_id):
    allowed = allowed_next_tokens(prev_token)

    if allowed is None:
        return logits

    filtered_logits = torch.full_like(logits, -1e9)

    for tok in allowed:
        if tok in token_to_id:
            filtered_logits[token_to_id[tok]] = logits[token_to_id[tok]]

    return filtered_logits


In [269]:
def generate_solution(
    model,
    input_text,
    token_to_id,
    id_to_token,
    max_new_tokens=30,
    temperature=.7,
    greedy=True
    filtered=True
):
    model.eval()

    tokens = algebra_splitter(input_text)

    for step in range(max_new_tokens):
        context_tokens = tokens[-model.context_length:]

        while len(context_tokens) < model.context_length:
            context_tokens = ["<PAD>"] + context_tokens

        context_ids = encode(context_tokens, token_to_id)
        x = torch.tensor([context_ids], dtype=torch.long).to(device)

        with torch.no_grad():
            preds = model(x)[0]

            prev_token = tokens[-1]

            if filtered:
              preds = filter_logits_by_allowed_tokens(
                  preds,
                  prev_token,
                  token_to_id
              )
            else:
              preds = preds

            if greedy:
                next_id = torch.argmax(preds).item()
            else:
                preds = preds / temperature
                probs = torch.softmax(preds, dim=-1)
                next_id = torch.multinomial(probs, 1).item()

        next_token = id_to_token[next_id]

        if next_token == "<EOS>":
            break

        tokens.append(next_token)

    return tokens

In [270]:
def tokens_to_string(tokens):
    filtered = [t for t in tokens if t not in {"<PAD>", "<SOS>", "<EOS>"}]
    return "".join(filtered)

In [271]:
generated = generate_solution(
    model,
    "x^2-4=8->",
    token_to_id,
    id_to_token,
    greedy=False
)

print(tokens_to_string(generated))

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
